# **--- WALMART PROJECT---** #

Walmart Inc. is an American multinational retail corporation that operates a chain of hypermarkets, discount department stores, and grocery stores from the United States. The goal of this project is to understand how the sales are influenced by economic indicators by using linear regression models and optimization.

## **I. Explanatory Data Analysis (EDA)** ##

### **1. Libraries import** ###

In [131]:
import pandas as pd
import numpy as np
import os
import json


from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats.mstats import winsorize

### **2. Data import and first observations** ###

In [132]:
raw_df = pd.read_csv("DATA/Data_raw/Walmart_Store_sales.csv")
raw_df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858
1,13.0,25-03-2011,1807545.43,0.0,42.38,3.435,128.616064,7.470
2,17.0,27-07-2012,NaN,0.0,NaN,NaN,130.719581,5.936
3,11.0,NaN,1244390.03,0.0,84.57,NaN,214.556497,7.346
4,6.0,28-05-2010,1644470.66,0.0,78.89,2.759,212.412888,7.092


In [133]:
print(f"The dataset has : \n - {raw_df.shape[0]} rows,\n - {raw_df.shape[1]} columns.")

The dataset has : 
 - 150 rows,
 - 8 columns.


In [134]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         150 non-null    float64
 1   Date          132 non-null    object 
 2   Weekly_Sales  136 non-null    float64
 3   Holiday_Flag  138 non-null    float64
 4   Temperature   132 non-null    float64
 5   Fuel_Price    136 non-null    float64
 6   CPI           138 non-null    float64
 7   Unemployment  135 non-null    float64
dtypes: float64(7), object(1)
memory usage: 9.5+ KB


We can observe that some columns have missing values. Moreover, the "Date" column will have to be transformed into a datetime object.

In [135]:
raw_df.describe(include="all")

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000000,132,1.360000e+02,138.000000,132.000000,136.000000,138.000000,135.000000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,19-10-2012,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.866667,NaN,1.249536e+06,0.079710,61.398106,3.320853,179.898509,7.598430
std,6.231191,NaN,6.474630e+05,0.271831,18.378901,0.478149,40.274956,1.577173
min,1.000000,NaN,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000
25%,4.000000,NaN,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500
50%,9.000000,NaN,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000
75%,15.750000,NaN,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000


In [136]:
raw_df["Store"].value_counts()

Store
3.0     15
1.0     11
18.0    10
19.0     9
5.0      9
14.0     9
13.0     9
7.0      8
17.0     8
2.0      8
8.0      8
6.0      7
20.0     7
4.0      7
12.0     5
10.0     5
15.0     4
16.0     4
9.0      4
11.0     3
Name: count, dtype: int64

The column "Store" does not correspond to unique identifiers for each store. We have several records for the same stores numbers so this column should not be deleted. But it is used to caracterize the number of the store so we will transform this variable as a categorical variable.

In [137]:
raw_df["Holiday_Flag"].value_counts()

Holiday_Flag
0.0    127
1.0     11
Name: count, dtype: int64

The variable Holiday_Flag correspond to a boolean variable so will have to change its type.

### **3. Prepare data for explanatory analysis** ###

**3.1. Types modification**

We have seen previously that "Holiday_Flag" needs to be considered as a boolean and "Date" as a datetime. Moreover, "Store" should be treated as a categorical variable.

In [138]:
# We transform the variable Holiday_Flag into a boolean
raw_df["Holiday_Flag"] = raw_df["Holiday_Flag"].astype("boolean")

# We transform the variable Date into a DateTime format
raw_df["Date"] = pd.to_datetime(raw_df["Date"], format="%d-%m-%Y")

# We transform "Store" as a categorical variable
raw_df["Store"] = raw_df["Store"].astype(int).astype("category")

**3.2 Separating variables according to types**

In [139]:
# Identify the target:
target="Weekly_Sales"
target_col = raw_df[target]

# Separating the variables per type :
num_col = raw_df.select_dtypes(include="float64").copy().drop(columns=target)
cat_col = raw_df.select_dtypes(include="category").copy()
bool_col = raw_df.select_dtypes(include="boolean").copy()
date_col = raw_df.select_dtypes(include="datetime64[ns]").copy()

# Lists of column names : 
num_col_names = num_col.columns.to_list()
cat_col_names = cat_col.columns.to_list()
bool_col_names = bool_col.columns.to_list()
date_col_names = date_col.columns.to_list()

**3.3. Analyzing duplicates**

In [140]:
# Check for duplicates
nb_duplicates = raw_df.duplicated().sum()
print(f"There are {nb_duplicates} duplicated rows in the dataset.")

There are 0 duplicated rows in the dataset.


**3.4. Analyzing missing values**

In [141]:
print("Number of missing values per variable : \n")
for col in raw_df.columns:
    nb_missing = raw_df[col].isna().sum()
    percentage_missing = round(100*nb_missing/raw_df.shape[0],2)
    print(f"- {col} : {nb_missing} missing values ({percentage_missing} % of total rows)")

Number of missing values per variable : 

- Store : 0 missing values (0.0 % of total rows)
- Date : 18 missing values (12.0 % of total rows)
- Weekly_Sales : 14 missing values (9.33 % of total rows)
- Holiday_Flag : 12 missing values (8.0 % of total rows)
- Temperature : 18 missing values (12.0 % of total rows)
- Fuel_Price : 14 missing values (9.33 % of total rows)
- CPI : 12 missing values (8.0 % of total rows)
- Unemployment : 15 missing values (10.0 % of total rows)


We observed missing values for all variables except `Store`. When possible, we will prefer imputation instead of deleting (small dataset and missing values <12%). We will adopt different strategies according to the variables: 

- **For the target**: We cannot use imputation techniques on the target (it might create some bias in the predictions) so we will drop all rows for which the value in `Weekly_Sales` is missing.

- **For the dates** (`Date`): Dates cannot be easily imputed so we will delete rows with empty dates.

- **For numerical features** (`Temperature`, `Fuel_Price`, `CPI`, `Unemployment`): use a SimpleImputer (with the median strategy : more robust than the mean),

- **For boolean features** (`Holiday_Flag`): Impute according to the most frequent value (mode).



In [142]:
# Drop rows for variables that cannot be imputed:
df = raw_df.dropna(subset=["Weekly_Sales"])
df = df.dropna(subset=["Date"])

nb_deleted = raw_df.shape[0]-df.shape[0]
print(f"After dropping: {df.shape[0]} remaining columns\n({nb_deleted} rows have been deleted : {round(100*nb_deleted/raw_df.shape[0],2)} % of the initial number of rows).")

After dropping: 118 remaining columns
(32 rows have been deleted : 21.33 % of the initial number of rows).


In [143]:
# Impute numerical features with the median:
imputer_num = SimpleImputer(strategy='median')
df[num_col_names] = imputer_num.fit_transform(df[num_col_names])

In [144]:
# Impute boolean feature with the mode:
imputer_bool = SimpleImputer(strategy='most_frequent')
df[bool_col_names] = imputer_bool.fit_transform(df[bool_col_names])

In [145]:
# Final check:
print(f"Number of missing values:\n{df.isnull().sum().sum()}")
print(f"\nFinal shape after dealing with missing values: {df.shape}")

Number of missing values:
0

Final shape after dealing with missing values: (118, 8)


Now `df` is complete without any missing values.

**3.5 Analyzing outliers**

In this project, will be considered as outliers all the numeric features that do not fall within the range : [Xˉ−3σ,Xˉ+3σ]). This concerns the columns : `Temperature`, `Fuel_price`, `CPI` and `Unemployment`.

In [146]:
for col in num_col_names:
    mean = df[col].mean()
    std = df[col].std()
    df = df[(df[col] >= mean - 3*std) & (df[col] <= mean + 3*std)]

In [147]:
print(f"\nFinal shape of the dataset after dealing with ouliers: {df.shape}.\n ")


Final shape of the dataset after dealing with ouliers: (113, 8).
 


**3.6 Create new features and drop useless column**

In [148]:
# We create new columns that might be useful to study the seasonal effects
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["DayOfWeek"] = df["Date"].dt.dayofweek

In [149]:
# We do not need the column 'Date' anymore, we can drop it:
df = df.drop("Date", axis=1)

**3.7. Save the cleaned dataset**

In [150]:
df.to_csv("DATA/Data_cleaned/Walmart_clean_dataset.csv")